In [1]:
import xml.etree.ElementTree as ET
import pandas as pd 
from datetime import datetime, timedelta

## Test Code

In [2]:
filename = "../data/02_23_26/02_23_26_07:00-stop-256.xml"

In [ ]:
text = pd.read_xml(filename)
text

In [ ]:
# check for available data
try:
    text_layer1 = pd.read_xml(filename, xpath=".//direction")
    print(text_layer1)

except Exception as e:
    print(f"No direction information found for route/time combination: {e}\nskipping...")


In [ ]:
tree = ET.parse(filename)
root = tree.getroot()

prediction_map = {prediction.get("routeTag"): prediction.text for prediction in root.findall(".//predictions")}
print(prediction_map)


text_layer2 = pd.read_xml(filename, xpath=".//prediction")
text_layer2

In [ ]:
rows = []
for prediction in root.findall(".//predictions"):
    route_title = prediction.get('routeTitle')
    route_tag = prediction.get('routeTag')
    stop_title = prediction.get('stopTitle')
    stop_tag = prediction.get('stopTag')
    direction = prediction.find("direction")

    if direction is not None:
        pred = direction.find('prediction')

        if pred is not None:
            rows.append({
                'route_title': route_title,
                'route_tag': route_tag,
                'stop_title': stop_title,
                'stop_tag': stop_tag,
                'direction': direction.get('title'),
                'mins': pred.get('minutes'),
                'secs': pred.get('seconds'),
                'epoch': pred.get('epochTime'),
                'is_departure': pred.get('isDeparture'),
                'affected_by_layover': pred.get('affectedByLayover'),
                'vehicle': pred.get('vehicle'),
                'block' : pred.get('block'),
                'trip_tag': pred.get('tripTag')
            })

pd.DataFrame(rows)

## Prediction Parsing Method

In [2]:
def parse_unitrans_predictions(file_path:str, base_time: datetime) -> pd.DataFrame:
    """
    Takes in the file path and the parsed base time from the filename
    Returns a df of the parsed xml file with additional features
    """

    # inital setup
    tree = ET.parse(file_path)
    root = tree.getroot()

    rows = []
    # loop through each prediction block
    for prediction in root.findall(".//predictions"):
        # get the header attributes
        route_title = prediction.get('routeTitle')
        route_tag = prediction.get('routeTag')
        stop_title = prediction.get('stopTitle')
        stop_tag = prediction.get('stopTag')
        direction = prediction.find("direction")

        # check the there is a valid direction
        if direction is not None:
            # if there is, grab the first one
            pred = direction.find('prediction')

            if pred is not None:
                # get the mins until arrival
                mins_val = int(pred.get('minutes', 0))

                # use the base time from the filename to calculate the arrival time
                predicted_arrival = base_time + timedelta(minutes=mins_val)
                
                # append all the features 
                rows.append({
                    'route_title': route_title,
                    'route_tag': route_tag,
                    'stop_title': stop_title,
                    'stop_tag': stop_tag,
                    'file_time': base_time,
                    'predicted_arrival_at': predicted_arrival,
                    'direction': direction.get('title').lower(),
                    'mins': pred.get('minutes'),
                    'secs': pred.get('seconds'),
                    'is_departure': pred.get('isDeparture'),
                    'affected_by_layover': pred.get('affectedByLayover'),
                    'vehicle': pred.get('vehicle'),
                    'block' : pred.get('block'),
                    'trip_tag': pred.get('tripTag')
                })

    df = pd.DataFrame(rows)
    if not df.empty:
        # set the arrival time to a datetime object
        df['predicted_arrival_at'] = pd.to_datetime(df['predicted_arrival_at'])
        
        # get the day of the week from the predicted arrival time
        df['day_name'] = df['predicted_arrival_at'].dt.day_name()

        # use the day name to assign the service class
        df['service_class'] = df['day_name'].apply(
            lambda day: 'MoTuTh' if day in ['Monday', 'Tuesday', 'Thursday'] else day
        )

        # ensure block is a string - for merging
        df['block'] = df['block'].astype(str)
    return df

In [3]:
import os

In [4]:
def extract_time_from_filename(filename):
    # extract the datetime from the filename
    base = os.path.basename(filename).split('-stop-')[0]
    return datetime.strptime(base, "%m_%d_%y_%H:%M")

In [5]:
# define all folders to process in one list
# list comprehension for the Feb dates + adding March 1st
folders = [f'../data/02_{i:02d}_26' for i in range(23, 29)] + ['../data/03_01_26']

all_dfs = []

for folder in folders:
    if not os.path.exists(folder):
        continue
        
    for file in os.listdir(folder):
        file_path = os.path.join(folder, file)
        
        try:
            file_dt = extract_time_from_filename(file)
            # ensure the xml is valid
            pd.read_xml(file_path, xpath=".//direction") 
            
            # parse and store the dataframe
            current_df = parse_unitrans_predictions(file_path=file_path, base_time=file_dt)
            all_dfs.append(current_df)
            
        except Exception:
            continue

# concatenate everything at once 
master_df = pd.concat(all_dfs, ignore_index=True) if all_dfs else pd.DataFrame()

In [6]:
master_df

,route_title,route_tag,stop_title,stop_tag,file_time,predicted_arrival_at,direction,mins,secs,is_departure,affected_by_layover,vehicle,block,trip_tag,day_name,service_class
0,A,A,Memorial Union & Main Island (SB),22273,2026-02-23 11:05:00,2026-02-23 12:09:00,outbound to el cemonte,64,3894,true,true,LMU_4019,29,A_11_outbound_1210,Monday,MoTuTh
1,G,G,Memorial Union & Main Island (SB),22273,2026-02-23 11:05:00,2026-02-23 11:24:00,outbound to n sycamore,19,1194,true,true,LMU_4088,24,G_11_outbound_1125,Monday,MoTuTh
2,M,M,Memorial Union & Main Island (SB),22273,2026-02-23 11:05:00,2026-02-23 11:24:00,outbound to drew,19,1194,true,true,LMU_4090,22,M_11_outbound_1125,Monday,MoTuTh
3,E,E,Memorial Union & East Island (SB),22274,2026-02-23 11:05:00,2026-02-23 11:29:00,outbound to j st,24,1494,true,true,LMU_4084,23,E_11_outbound_1130,Monday,MoTuTh
4,F,F,Memorial Union & East Island (SB),22274,2026-02-23 11:05:00,2026-02-23 11:24:00,outbound to f st,19,1194,true,true,LMU_4087,25,F_11_outbound_1125,Monday,MoTuTh
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
15452,U,U,Memorial Union & East Island (SB),22274,2026-03-01 18:25:00,2026-03-01 18:34:00,outbound to west village via russell blvd,9,596,true,true,LMU_4011,65,U_20_outbound_1835,Sunday,Sunday
15453,M,M,Memorial Union & Main Island (SB),22273,2026-03-01 18:30:00,2026-03-01 18:34:00,outbound to drew,4,296,true,true,LMU_4020,64,M_20_outbound_1835,Sunday,Sunday
15454,U,U,Memorial Union & East Island (SB),22274,2026-03-01 18:30:00,2026-03-01 18:35:00,outbound to west village via russell blvd,5,339,true,true,LMU_4011,65,U_20_outbound_1835,Sunday,Sunday
15455,M,M,Memorial Union & Main Island (SB),22273,2026-03-01 18:35:00,2026-03-01 18:35:00,outbound to drew,0,40,true,true,LMU_4020,64,M_20_outbound_1835,Sunday,Sunday


In [7]:
import sqlite3

In [8]:
# connect to existing database
conn = sqlite3.connect('unitrans.db')

# create predictions table
master_df.to_sql('predictions', conn, if_exists='replace', index=False)

pd.read_sql_query("SELECT * FROM predictions", conn)

,route_title,route_tag,stop_title,stop_tag,file_time,predicted_arrival_at,direction,mins,secs,is_departure,affected_by_layover,vehicle,block,trip_tag,day_name,service_class
0,A,A,Memorial Union & Main Island (SB),22273,2026-02-23 11:05:00,2026-02-23 12:09:00,outbound to el cemonte,64,3894,true,true,LMU_4019,29,A_11_outbound_1210,Monday,MoTuTh
1,G,G,Memorial Union & Main Island (SB),22273,2026-02-23 11:05:00,2026-02-23 11:24:00,outbound to n sycamore,19,1194,true,true,LMU_4088,24,G_11_outbound_1125,Monday,MoTuTh
2,M,M,Memorial Union & Main Island (SB),22273,2026-02-23 11:05:00,2026-02-23 11:24:00,outbound to drew,19,1194,true,true,LMU_4090,22,M_11_outbound_1125,Monday,MoTuTh
3,E,E,Memorial Union & East Island (SB),22274,2026-02-23 11:05:00,2026-02-23 11:29:00,outbound to j st,24,1494,true,true,LMU_4084,23,E_11_outbound_1130,Monday,MoTuTh
4,F,F,Memorial Union & East Island (SB),22274,2026-02-23 11:05:00,2026-02-23 11:24:00,outbound to f st,19,1194,true,true,LMU_4087,25,F_11_outbound_1125,Monday,MoTuTh
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
15452,U,U,Memorial Union & East Island (SB),22274,2026-03-01 18:25:00,2026-03-01 18:34:00,outbound to west village via russell blvd,9,596,true,true,LMU_4011,65,U_20_outbound_1835,Sunday,Sunday
15453,M,M,Memorial Union & Main Island (SB),22273,2026-03-01 18:30:00,2026-03-01 18:34:00,outbound to drew,4,296,true,true,LMU_4020,64,M_20_outbound_1835,Sunday,Sunday
15454,U,U,Memorial Union & East Island (SB),22274,2026-03-01 18:30:00,2026-03-01 18:35:00,outbound to west village via russell blvd,5,339,true,true,LMU_4011,65,U_20_outbound_1835,Sunday,Sunday
15455,M,M,Memorial Union & Main Island (SB),22273,2026-03-01 18:35:00,2026-03-01 18:35:00,outbound to drew,0,40,true,true,LMU_4020,64,M_20_outbound_1835,Sunday,Sunday
